# Dataset Preparation

In [ ]:

from pathlib import Path
import json
import os
import random
import subprocess
from urllib.parse import quote

import lizard
import pandas as pd
from pygments import lex
from pygments.lexers import get_lexer_for_filename
from pygments.token import Token

SEED = 42
FILES_PER_REPOSITORY = 200
EXPECTED_EVALUATION_SIZE = 2578

ROOT = Path(".")
REPOS_DIR = ROOT / "repos"
REFERENCE_FILE = ROOT / "reference_summaries.csv"
DATASET_FILE = ROOT / "evaluation_dataset.jsonl"
MANIFEST_FILE = ROOT / "evaluation_manifest.csv"
REPOSITORY_MANIFEST_FILE = ROOT / "repository_manifest.csv"

REPOSITORIES = {
    "storm": "https://github.com/apache/storm.git",
    "butterknife": "https://github.com/JakeWharton/butterknife.git",
    "crate": "https://github.com/crate/crate.git",
    "hystrix": "https://github.com/Netflix/Hystrix.git",
    "jabref": "https://github.com/JabRef/jabref.git",
    "jcabi": "https://github.com/jcabi/jcabi-github.git",
    "openmicroscopy": "https://github.com/ome/openmicroscopy.git",
    "presto": "https://github.com/prestodb/presto.git",
    "rxandroid": "https://github.com/ReactiveX/RxAndroid.git",
    "sponge": "https://github.com/SpongePowered/SpongeAPI.git",
    "springboot": "https://github.com/spring-projects/spring-boot.git",
    "okhttp": "https://github.com/square/okhttp.git",
    "retrofit": "https://github.com/square/retrofit.git",
    "wordpress": "https://github.com/wordpress-mobile/WordPress-Android.git",
}

SOURCE_EXTENSIONS = {".py", ".java", ".kt", ".js"}


In [ ]:

def run_git(repo_path, *args):
    result = subprocess.run(
        ["git", "-C", str(repo_path), *args],
        check=True,
        capture_output=True,
        text=True,
    )
    return result.stdout.strip()


def prepare_repository(name, url):
    repo_path = REPOS_DIR / name

    if not repo_path.exists():
        subprocess.run(
            ["git", "clone", "--filter=blob:none", url, str(repo_path)],
            check=True,
        )

    shallow = run_git(repo_path, "rev-parse", "--is-shallow-repository")
    if shallow.lower() == "true":
        subprocess.run(
            ["git", "-C", str(repo_path), "fetch", "--unshallow", "--tags", "--prune"],
            check=True,
        )

    return {
        "path": repo_path,
        "sha": run_git(repo_path, "rev-parse", "HEAD"),
        "remote": run_git(repo_path, "remote", "get-url", "origin"),
    }


def github_url(remote):
    if remote.startswith("git@github.com:"):
        remote = "https://github.com/" + remote.split("git@github.com:", 1)[1]
    return remote[:-4] if remote.endswith(".git") else remote


def list_source_files(repo_path):
    files = []
    for path in repo_path.rglob("*"):
        if (
            path.is_file()
            and ".git" not in path.parts
            and path.suffix.lower() in SOURCE_EXTENSIONS
        ):
            files.append(path.relative_to(repo_path).as_posix())
    return sorted(files)


def read_text(path):
    return path.read_text(encoding="utf-8", errors="ignore")


def read_readme(repo_path):
    candidates = [
        path
        for path in repo_path.iterdir()
        if path.is_file() and path.name.lower().startswith("readme")
    ]

    if not candidates:
        return "", ""

    candidates.sort(key=lambda path: path.name.lower())
    selected = candidates[0]
    return selected.name, read_text(selected)


def extract_comments(file_path, source):
    try:
        lexer = get_lexer_for_filename(file_path.name, source)
    except Exception:
        return ""

    values = []
    for token_type, value in lex(source, lexer):
        if token_type in Token.Comment or token_type in Token.Literal.String.Doc:
            value = value.strip()
            if value:
                values.append(value)

    return "\n".join(values)


def calculate_complexity(file_path):
    analysis = lizard.analyze_file(str(file_path))
    functions = analysis.function_list
    values = [item.cyclomatic_complexity for item in functions]

    return {
        "nloc": int(analysis.nloc),
        "function_count": len(functions),
        "mean_ccn": round(sum(values) / len(values), 4) if values else 0.0,
        "max_ccn": int(max(values)) if values else 0,
    }


def file_commit_history(repo_path, relative_path):
    output = run_git(
        repo_path,
        "log",
        "--follow",
        "--format=%H%x09%ad%x09%s",
        "--date=short",
        "--",
        relative_path,
    )

    commits = []
    for line in output.splitlines():
        fields = line.split("\t", 2)
        if len(fields) == 3:
            commit_hash, date, subject = fields
            commits.append({
                "hash": commit_hash,
                "date": date,
                "subject": subject,
            })

    return commits


def build_record(repository, repository_info, relative_path, reference_summary):
    repo_path = repository_info["path"]
    file_path = repo_path / relative_path
    source = read_text(file_path)
    readme_name, readme_content = read_readme(repo_path)
    comments = extract_comments(file_path, source)
    complexity = calculate_complexity(file_path)
    commits = file_commit_history(repo_path, relative_path)

    lines = source.splitlines()
    base_url = github_url(repository_info["remote"])
    file_url = (
        f"{base_url}/blob/{repository_info['sha']}/"
        f"{quote(relative_path, safe='/')}"
    )

    return {
        "file_id": f"{repository}:{relative_path}",
        "repository": repository,
        "repository_sha": repository_info["sha"],
        "file_name": relative_path,
        "file_url": file_url,
        "source_code": source,
        "code_size": {
            "lines": len(lines),
            "nonblank_lines": sum(bool(line.strip()) for line in lines),
            "characters": len(source),
            "bytes": len(source.encode("utf-8")),
            "nloc": complexity["nloc"],
        },
        "comments": comments,
        "readme_name": readme_name,
        "readme_content": readme_content,
        "complexity": complexity,
        "commits": commits,
        "reference_summary": reference_summary,
    }


In [ ]:

REPOS_DIR.mkdir(exist_ok=True)

repository_info = {}
repository_manifest_rows = []
candidate_rows = []

for name, url in REPOSITORIES.items():
    info = prepare_repository(name, url)
    repository_info[name] = info

    repository_manifest_rows.append({
        "Repository": name,
        "URL": github_url(info["remote"]),
        "Commit SHA": info["sha"],
    })

    for relative_path in list_source_files(info["path"]):
        candidate_rows.append({
            "Repository": name,
            "File Name": relative_path,
        })

pd.DataFrame(repository_manifest_rows).to_csv(
    REPOSITORY_MANIFEST_FILE,
    index=False,
)

candidates = pd.DataFrame(candidate_rows)

if not REFERENCE_FILE.exists():
    template = candidates.copy()
    template["Reference Summary"] = ""
    template.to_csv("reference_summaries_template.csv", index=False)
    raise FileNotFoundError(
        "reference_summaries.csv is required before the evaluation dataset can be built."
    )

references = pd.read_csv(REFERENCE_FILE)
references["Reference Summary"] = (
    references["Reference Summary"]
    .fillna("")
    .astype(str)
    .str.strip()
)

eligible = candidates.merge(
    references[
        references["Reference Summary"] != ""
    ][["Repository", "File Name", "Reference Summary"]],
    on=["Repository", "File Name"],
    how="inner",
)

selected_groups = []

for repository, group in eligible.groupby("Repository", sort=True):
    sample_size = min(FILES_PER_REPOSITORY, len(group))
    selected_groups.append(
        group.sample(n=sample_size, random_state=SEED)
    )

selected = (
    pd.concat(selected_groups, ignore_index=True)
    .sort_values(["Repository", "File Name"])
    .reset_index(drop=True)
)

if len(selected) != EXPECTED_EVALUATION_SIZE:
    raise ValueError(
        f"Expected {EXPECTED_EVALUATION_SIZE} evaluation files, "
        f"but {len(selected)} files were selected. "
        "Check reference_summaries.csv and repository contents."
    )

records = []

for _, row in selected.iterrows():
    records.append(
        build_record(
            repository=row["Repository"],
            repository_info=repository_info[row["Repository"]],
            relative_path=row["File Name"],
            reference_summary=row["Reference Summary"],
        )
    )

with DATASET_FILE.open("w", encoding="utf-8") as file:
    for record in records:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")

manifest = pd.DataFrame([
    {
        "File ID": record["file_id"],
        "Repository": record["repository"],
        "Repository SHA": record["repository_sha"],
        "File Name": record["file_name"],
        "File URL": record["file_url"],
        "Code Lines": record["code_size"]["lines"],
        "Code Nonblank Lines": record["code_size"]["nonblank_lines"],
        "Code Characters": record["code_size"]["characters"],
        "Code Bytes": record["code_size"]["bytes"],
        "NLOC": record["code_size"]["nloc"],
        "Function Count": record["complexity"]["function_count"],
        "Mean CCN": record["complexity"]["mean_ccn"],
        "Max CCN": record["complexity"]["max_ccn"],
        "Comment Characters": len(record["comments"]),
        "README Characters": len(record["readme_content"]),
        "Commit Count": len(record["commits"]),
        "Reference Summary": record["reference_summary"],
    }
    for record in records
])

manifest.to_csv(MANIFEST_FILE, index=False)
display(manifest.head())
